In [ ]:
# TemporalBench v2 — Overlap Tier
# Difficulty: Overlapping validity windows, PastQueryTrap, CurrentQuery
# Task families: PastQueryTrap, ChangeDetection, CurrentQuery, CausalQuery

import json
import numpy as np
from typing import Dict, List, Any, Optional

# @kbench.task(name="TemporalBench v2", description="Overlap conditions — PastQueryTrap, ChangeDetection, CurrentQuery", tasks=4)

class Fact:
    def __init__(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        self.key = key
        self.value = value
        self.valid_from = valid_from
        self.valid_to = valid_to
        self.accesses = accesses
    
    def is_valid_at(self, day: int) -> bool:
        return self.valid_from <= day < self.valid_to
    
    def age(self, day: int) -> float:
        return day - self.valid_from
    
    def decay_score(self, day: int, half_life: float = 50.0) -> float:
        return np.exp(-0.693 * self.age(day) / half_life)
    
    def attention_score(self) -> float:
        return 1.0 + 0.001 * np.log1p(self.accesses)


class TemporalAttentionStore:
    """System D — decay + attention baseline."""
    def __init__(self, half_life: float = 50.0, temporal_weight: float = 0.9, attention_weight: float = 0.1):
        self.half_life = half_life
        self.temporal_weight = temporal_weight
        self.attention_weight = attention_weight
        self.facts: Dict[str, List[Fact]] = {}
    
    def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))
    
    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        scores = [(self.temporal_weight * f.decay_score(as_of_day, self.half_life) +
                   self.attention_weight * f.attention_score(), f) for f in candidates]
        return max(scores, key=lambda x: x[0])[1].value


class TemporalAttentionStoreWithValidity:
    """System D_revised — validity windows as hard gate."""
    def __init__(self, half_life: float = 50.0):
        self.half_life = half_life
        self.facts: Dict[str, List[Fact]] = {}
    
    def put(self, key: str, value: str, valid_from: int, valid_to: int, accesses: int = 0):
        if key not in self.facts:
            self.facts[key] = []
        self.facts[key].append(Fact(key, value, valid_from, valid_to, accesses))
    
    def get(self, key: str, as_of_day: Optional[int] = None) -> Optional[str]:
        if key not in self.facts:
            return None
        # HARD validity gate
        candidates = [f for f in self.facts[key] if f.is_valid_at(as_of_day)]
        if not candidates:
            return None
        scores = [(f.decay_score(as_of_day, self.half_life) * f.attention_score(), f) for f in candidates]
        return max(scores, key=lambda x: x[0])[1].value


def load_benchmark(path: str):
    with open(path + '_questions.jsonl') as f:
        questions = [json.loads(line) for line in f]
    with open(path + '_facts.jsonl') as f:
        facts_data = [json.loads(line) for line in f]
    with open(path + '_events.jsonl') as f:
        events_data = [json.loads(line) for line in f]
    return questions, facts_data, events_data


def build_store(facts_data, events_data, store_class):
    store = store_class()
    for fact in facts_data:
        store.put(fact['key'], fact['value'], fact['valid_from'], fact['valid_to'],
                  fact.get('accesses', 0))
    event_log = sorted(events_data, key=lambda x: x['day'])
    return store, event_log


def score_question(question: Dict, predicted: Optional[str]) -> float:
    task_family = question.get('task_family', 'Unknown')
    answer = question.get('answer') or question.get('expected_answer') or question.get('expected_day', '')
    
    if not answer:
        return 0.0 if predicted else 1.0  # pass on no-answer questions
    
    pred = str(predicted).strip().lower() if predicted else ''
    exp = str(answer).strip().lower()
    
    if task_family == 'CurrentQuery':
        # CurrentState: is fact X still valid? Binary yes/no
        return 1.0 if (('yes' in pred and 'yes' in exp) or ('no' in pred and 'no' in exp)
                       or pred == exp) else 0.0
    
    # Default: exact match
    return 1.0 if pred == exp else 0.0


def run_evaluation(questions, facts_data, events_data, store_class):
    store, event_log = build_store(facts_data, events_data, store_class)
    
    results = {}
    for q in questions:
        tf = q.get('task_family', 'Unknown')
        if tf not in results:
            results[tf] = {'correct': 0.0, 'total': 0}
        
        as_of_day = q.get('as_of_day') or q.get('day')
        key = f"{q.get('domain', '')}:{q.get('subject', '')}"
        
        # Replay events up to query day
        for ev in event_log:
            if ev['day'] > as_of_day:
                break
            if ev.get('type') == 'fact_access':
                for kfs in store.facts.values():
                    for f in kfs:
                        if f.valid_from == ev.get('valid_from') and f.valid_to == ev.get('valid_to'):
                            f.accesses = ev.get('accesses', f.accesses)
        
        predicted = store.get(key, as_of_day)
        score = score_question(q, predicted)
        results[tf]['correct'] += score
        results[tf]['total'] += 1
    
    metrics = {tf: r['correct'] / r['total'] if r['total'] > 0 else 0.0 for tf, r in results.items()}
    trs = np.mean(list(metrics.values())) if metrics else 0.0
    return {'metrics': metrics, 'trs': trs}


DATA_PATH = '/kaggle/input/temporalbench-v2/temporalbench_v2'

print('Loading TemporalBench v2 (overlap tier)...')
questions, facts_data, events_data = load_benchmark(DATA_PATH)
print(f'Loaded {len(questions)} questions, {len(facts_data)} facts, {len(events_data)} events')

print('\nEvaluating D_revised (Validity Windows)...')
results_revised = run_evaluation(questions, facts_data, events_data, TemporalAttentionStoreWithValidity)
print(f'TRS: {results_revised["trs"]:.4f}')
for tf, acc in results_revised['metrics'].items():
    print(f'  {tf}: {acc:.1%}')

print('\nEvaluating D (Decay baseline)...')
results_d = run_evaluation(questions, facts_data, events_data, TemporalAttentionStore)
print(f'TRS: {results_d["trs"]:.4f}')
for tf, acc in results_d['metrics'].items():
    print(f'  {tf}: {acc:.1%}')

print('\n--- SUMMARY ---')
print(f'D_revised (Validity Windows): {results_revised["trs"]:.1%}')
print(f'D (Decay baseline):           {results_d["trs"]:.1%}')
print(f'Gap closed:                  {(results_revised["trs"] - results_d["trs"]):.1%}')
